In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pymilvus import MilvusClient, DataType

# Đường dẫn dữ liệu
CLIP_DIR = "data/clip_features"
CSV_DIR = "data/csv_metadata"
MILVUS_URI = "./milvus_demo.db"     
COLLECTION_NAME = "clip_keyframes"

def normalize(vectors: np.ndarray) -> np.ndarray:
    """Chuẩn hóa L2 cho mảng vectors."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return vectors / norms

def create_collection(client: MilvusClient, dim: int, fresh: bool = False):
    """Khởi tạo collection với số chiều vector được truyền động."""
    if client.has_collection(COLLECTION_NAME):
        if fresh:
            print(f"[-] Xóa collection cũ: {COLLECTION_NAME}")
            client.drop_collection(COLLECTION_NAME)
        else:
            return

    schema = client.create_schema(auto_id=True, enable_dynamic_field=False)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="video_id", datatype=DataType.VARCHAR, max_length=256)
    schema.add_field(field_name="frame_id", datatype=DataType.INT64)
    schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=dim)

    # Cấu hình Index: Dùng FLAT cho Milvus Lite local để không đòi faiss-cpu
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_type="FLAT",          # Đổi thành "HNSW" nếu chạy server Milvus lớn
        metric_type="IP",           # Inner Product (Cosine similarity khi vector đã L2-normalized)
        params={},
    )

    client.create_collection(
        collection_name=COLLECTION_NAME,
        schema=schema,
        index_params=index_params,
    )
    print(f"[+] Đã tạo collection mới: {COLLECTION_NAME} (dim={dim})")

def get_existing_videos(client: MilvusClient) -> set:
    """Lấy danh sách video_id đã có trong collection để tránh nạp trùng."""
    existing = set()
    try:
        iterator = client.query_iterator(
            collection_name=COLLECTION_NAME,
            filter="",
            output_fields=["video_id"],
            batch_size=1000,
        )
        while True:
            batch = iterator.next()
            if not batch:
                break
            existing.update(row["video_id"] for row in batch)
        iterator.close()
    except Exception:
        pass
    return existing

def insert_video(client: MilvusClient, npy_file: str) -> int:
    """Nạp vector và metadata của 1 video vào Milvus."""
    video_id = os.path.splitext(npy_file)[0]
    npy_path = os.path.join(CLIP_DIR, npy_file)
    csv_path = os.path.join(CSV_DIR, f"{video_id}.csv")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy CSV tương ứng: {csv_path}")

    # Đọc vector
    features = np.load(npy_path).astype(np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    
    # Đọc CSV metadata và xử lý frame_idx linh hoạt
    df = pd.read_csv(csv_path)
    if "frame_idx" in df.columns:
        frame_ids = df["frame_idx"].astype(int).tolist()
    elif "frame_id" in df.columns:
        frame_ids = df["frame_id"].astype(int).tolist()
    else:
        frame_ids = list(range(1, len(df) + 1))

    if len(features) != len(frame_ids):
        raise ValueError(
            f"{video_id}: Lệch kích thước ({len(features)} vectors != {len(frame_ids)} frames)"
        )

    features = normalize(features)

    rows = [
        {
            "video_id": video_id,
            "frame_id": frame_ids[i],
            "embedding": features[i].tolist(),
        }
        for i in range(len(features))
    ]

    result = client.insert(collection_name=COLLECTION_NAME, data=rows)
    print(f"  + {video_id}: Đã nạp {result['insert_count']} vectors")
    return result["insert_count"]

def run(mode: str = "build"):
    if not os.path.exists(CLIP_DIR):
        raise FileNotFoundError(f"Thư mục '{CLIP_DIR}' không tồn tại.")

    npy_files = sorted(f for f in os.listdir(CLIP_DIR) if f.endswith(".npy"))
    if not npy_files:
        raise RuntimeError(f"Không tìm thấy file .npy nào trong thư mục '{CLIP_DIR}'")

    # Tự động lấy số chiều vector từ file npy đầu tiên
    first_npy = np.load(os.path.join(CLIP_DIR, npy_files[0]))
    dim = first_npy.shape[-1]

    client = MilvusClient(uri=MILVUS_URI)
    create_collection(client, dim=dim, fresh=(mode == "build"))

    existing_videos = get_existing_videos(client) if mode == "rebuild" else set()

    total, added, skipped = 0, 0, 0
    print(f"\n--- BẮT ĐẦU NẠP DỮ LIỆU (Mode: {mode}) ---")
    for npy_file in npy_files:
        video_id = os.path.splitext(npy_file)[0]
        if video_id in existing_videos:
            print(f"  - Skip: {video_id} (Đã có sẵn)")
            skipped += 1
            continue
        total += insert_video(client, npy_file)
        added += 1

    # Nạp collection vào bộ nhớ để sẵn sàng search
    client.flush(collection_name=COLLECTION_NAME)
    client.load_collection(collection_name=COLLECTION_NAME)

    print(f"\n Hoàn tất nạp dữ liệu!")
    print(f"Tổng video nạp mới : {added} ({total} vectors)")
    if mode == "rebuild":
        print(f"Tổng video bỏ qua  : {skipped}")

if __name__ == "__main__":
    run(mode="build")

[-] Xóa collection cũ: clip_keyframes
[+] Đã tạo collection mới: clip_keyframes (dim=512)

--- BẮT ĐẦU NẠP DỮ LIỆU (Mode: build) ---
  + video_01: Đã nạp 307 vectors
  + video_02: Đã nạp 200 vectors
  + video_03: Đã nạp 120 vectors

 Hoàn tất nạp dữ liệu!
Tổng video nạp mới : 3 (627 vectors)


In [2]:
import numpy as np
from trake_retriever import TrakeEngine

# Truyền rõ đường dẫn file database và collection đã nạp ở Cell 1
engine = TrakeEngine(
    milvus_uri="./milvus_demo.db",
    collection_name="clip_keyframes"
)

# Kiểm tra encode thử 1 câu
emb = engine._encode_text("a man kicking a soccer ball")
print("Embedding shape:", emb.shape)
print("Norm xấp xỉ 1.0:", np.linalg.norm(emb))

c:\conda\Miniconda3\envs\vlm\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Embedding shape: (512,)
Norm xấp xỉ 1.0: 0.99999994


In [ ]:
from aic_agent_core import route_query, TaskType
from trake_retriever import TrakeEngine

# 1. Khởi tạo Engine 
engine = TrakeEngine(
    milvus_uri="./milvus_demo.db",
    collection_name="clip_keyframes",
    csv_dir="data/csv_metadata",
    feature_dir="data/clip_features"
)

raw_query = (
    "First is the sunset city skyline with 60 seconds news logo, "
    "then two news anchors male and female presenting in newsroom studio, "
    "after that an aerial drone shot of collapsed asphalt road falling into river water, "
    "and finally a white car driving into hospital entrance."
)

task_type, structured_query = route_query(raw_query)
print(f"Task Type: {task_type}\n")

if task_type == TaskType.TRAKE:
    best_video, keyframe_ids = engine.solve_trake(
        structured_query, 
        top_k_videos=5, 
        min_gap=1
    )

    print("================ KẾT QUẢ TRAKE ================")
    print(f"Video khớp nhất: {best_video}")
    print(f"Aligned Frame IDs: {keyframe_ids}\n")
    
    if keyframe_ids:
        for i, (ev, f_id) in enumerate(zip(structured_query.events, keyframe_ids), 1):
            print(f"Event {i}: \"{ev.description}\" : Khớp Frame ID: {f_id}")
    else:
        print("Không tìm thấy chuỗi frame thỏa mãn ràng buộc.")

Task Type: TRAKE

================ KẾT QUẢ TRAKE ================
Video khớp nhất: video_01
Aligned Frame IDs: [1, 12, 30, 67]

  📌 Event 1: "Sunset city skyline with 60 seconds news logo" ➔ Khớp Frame ID: 1
  📌 Event 2: "Two news anchors male and female presenting in newsroom studio" ➔ Khớp Frame ID: 12
  📌 Event 3: "Aerial drone shot of collapsed asphalt road falling into river water" ➔ Khớp Frame ID: 30
  📌 Event 4: "White car driving into hospital entrance" ➔ Khớp Frame ID: 67
